# Planejamento e Reflexão

O laço do notebook anterior decide um passo de cada vez. Ele olha a última observação, escolhe a ferramenta seguinte e repete, sem escrever em lugar nenhum a sequência inteira que a tarefa exige.

Duas técnicas acrescentam esse registro, em momentos diferentes. Planejar é escrever os passos antes de agir e guardá-los onde o programa possa lê-los. Refletir é avaliar a resposta depois de pronta, segundo uma rubrica, e reescrevê-la enquanto a avaliação pedir.

As duas aparecem aqui sobre a mesma tarefa. O plano, a decisão de replanejar, a extração dos números e a crítica são saída estruturada, cada um com o seu esquema Pydantic.

In [ ]:
# No Google Colab, descomente e rode uma vez (Ambiente de execução > GPU).
# !pip install -q "agentkit @ git+https://github.com/silvaan/agentic-ai"

from pathlib import Path
from typing import Literal

import pandas as pd
import torch
from pydantic import BaseModel, Field

from agentkit import LLM, Agent, tool

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
llm = LLM(MODEL_NAME, device=device, temperature=0.0, max_tokens=250)
print(llm.model)

## A tarefa

Somar o total de entregas dos relatórios mensais e gravar o resultado em `total.txt`. São quatro etapas: listar os arquivos, ler cada um, somar os valores e gravar o total.

Os três relatórios são criados agora, e a soma correta fica guardada para conferência no fim. A pasta é esvaziada antes, porque o notebook também grava dentro dela.

In [ ]:
REPORTS = Path("workspace/reports")
REPORTS.mkdir(parents=True, exist_ok=True)
for path in REPORTS.iterdir():
    path.unlink()

MONTHS = {"janeiro.txt": 1200, "fevereiro.txt": 950, "marco.txt": 1430}
for name, value in MONTHS.items():
    (REPORTS / name).write_text(f"Relatorio mensal\nTotal de entregas: {value}\n", encoding="utf-8")

ANSWER = sum(MONTHS.values())
print("arquivos:", sorted(path.name for path in REPORTS.iterdir()))
print("soma correta:", ANSWER)

As três ferramentas são as únicas ações disponíveis ao agente: listar a pasta, ler um arquivo e escrever um arquivo. Nenhuma delas soma.

In [ ]:
@tool
def list_files() -> str:
    """Lista os arquivos disponíveis na pasta de relatórios."""
    return ", ".join(sorted(path.name for path in REPORTS.iterdir()))


@tool
def read_file(name: str) -> str:
    """Lê um arquivo da pasta de relatórios e devolve o conteúdo."""
    return (REPORTS / name).read_text(encoding="utf-8")


@tool
def write_file(name: str, content: str) -> str:
    """Escreve um arquivo na pasta de relatórios."""
    (REPORTS / name).write_text(content, encoding="utf-8")
    return f"{name} gravado"


TOOLS = [list_files, read_file, write_file]

TASK = ("Leia o total de entregas de cada arquivo da pasta de relatórios, "
        "some os valores e grave o resultado em total.txt.")

## ReAct

ReAct, de *reasoning and acting*, é o nome do laço construído no notebook de ferramentas. O modelo pensa e pede uma ferramenta, o programa executa e devolve a observação, e o modelo pensa de novo com ela no contexto. O ciclo termina quando a resposta vem sem pedido de ferramenta.

`run_react` é esse laço escrito por extenso, com uma linha para cada uma das três etapas. É o mesmo que o `Agent` do `agentkit` faz, aberto aqui porque a tarefa desta aula é justamente o que falta nele.

In [ ]:
def run_react(task: str, max_steps: int = 6) -> list[dict]:
    """Alterna raciocínio, chamada de ferramenta e observação até a resposta final."""
    llm_with_tools = llm.bind_tools(TOOLS)
    tools_by_name = {fn.tool_schema["name"]: fn for fn in TOOLS}
    messages = [{"role": "user", "content": task}]

    for _ in range(max_steps):
        message = llm_with_tools.invoke(messages)                 # pensar
        messages.append(message)

        if "tool_calls" not in message:                           # resposta final: o laço acaba
            break

        for call in message["tool_calls"]:                        # agir
            observation = str(tools_by_name[call["name"]](**call["arguments"]))
            messages.append({"role": "tool",                      # observar
                             "name": call["name"],
                             "content": observation})

    return messages

In [ ]:
messages = run_react(TASK)

for message in messages:
    print(message["role"], ":", message.get("content") or message["tool_calls"])

O agente listou a pasta e parou. Os três nomes de arquivo estavam na observação que ele acabara de receber e, mesmo assim, ele não chamou `read_file`: pediu ao usuário o caminho dos arquivos, e `total.txt` nunca foi criado.

Quatro mensagens para uma tarefa de quatro etapas, e a execução terminou na primeira. O laço escolhe bem o passo imediato, porque a última observação basta para escolher, e não guarda nada sobre as etapas que continuam pendentes. É esse registro que falta.

### Exercício 1

A secretaria do curso reclama que o agente para no meio do serviço, e alguém sugere escrever o pedido já dividido em passos numerados: listar a pasta, ler cada arquivo, somar os valores e gravar o total. Entregue esse pedido a `run_react`, e responda até que passo ele chegou e se escrever o plano dentro do pedido foi suficiente.

In [ ]:
# Seu código aqui

## Planejamento

Planejar é escrever a sequência de passos antes de executar qualquer um deles, e guardá-la em uma estrutura que o programa consiga percorrer, marcar e refazer.

O esquema `Plan` é essa estrutura: uma lista de passos, cada passo em uma frase. Ele serve às duas formas de obter um plano que vêm a seguir, pedir ao modelo e montar por código.

In [ ]:
class Plan(BaseModel):
    steps: list[str] = Field(min_length=1, max_length=5)

### Plano escrito pelo modelo

A primeira forma é pedir o plano ao modelo. Gerada sob o esquema `Plan`, a resposta chega como lista de passos, e não como parágrafo.

O prompt é um gabarito de texto com as lacunas preenchidas na hora da chamada, como o gabarito da mensagem de sistema do notebook de memória.

In [ ]:
PLAN_PROMPT = """Tarefa: {task}
Ferramentas disponíveis: {tools}

Escreva os passos necessários, em português, um passo por frase."""


def plan_from_model() -> Plan:
    """Pede um plano ao modelo, validado pelo esquema Plan."""
    prompt = PLAN_PROMPT.format(
        task=TASK,
        tools=", ".join(fn.tool_schema["name"] for fn in TOOLS),
    )
    return llm.generate_structured([{"role": "user", "content": prompt}], Plan, max_tokens=200)


guessed = plan_from_model()
pd.DataFrame({"passo": guessed.steps})

O esquema fez o seu trabalho: o que voltou é uma lista de passos que o programa consegue percorrer.

O conteúdo é outra história. O primeiro passo manda importar bibliotecas, coisa que ferramenta nenhuma faz, e o segundo para na listagem da pasta. Ler os arquivos, somar e gravar não aparecem. O modelo escreveu o plano sem saber quantos arquivos existem, porque ninguém olhou a pasta ainda.

### Plano montado por código

A quantidade de passos depende de quantos arquivos existem na pasta, e isso não é palpite: é uma chamada de ferramenta.

`plan_from_files` devolve um `Plan` como o anterior, do mesmo tipo e sujeito à mesma validação, preenchido com um passo de leitura por arquivo encontrado.

In [ ]:
def plan_from_files() -> Plan:
    """Monta um plano com um passo de leitura para cada arquivo da pasta."""
    steps = []
    for name in list_files().split(", "):
        steps.append(f"Leia o arquivo {name} e informe o total de entregas.")
    return Plan(steps=steps)


plan = plan_from_files()
pd.DataFrame({"passo": plan.steps})

Os dois planos têm o mesmo tipo e vieram de lugares diferentes. Neste, cada passo é uma instrução que uma única chamada de ferramenta resolve, e a quantidade de passos veio da pasta.

A divisão de responsabilidade é a ideia da seção: o código decide **quantos** passos existem, porque isso é contagem, e o modelo decide **o que fazer** dentro de cada passo, porque isso é interpretação.

### Execução do plano

Executar o plano é um laço `for` sobre os passos. Cada passo vira a tarefa de um `Agent`, o mesmo laço da seção anterior, que recebe uma instrução curta e não sabe do resto do plano. A resposta final de cada agente fica guardada como observação.

In [ ]:
def run_plan(plan: Plan) -> list[str]:
    """Executa um passo por vez e devolve a observação de cada um."""
    observations = []
    for step in plan.steps:
        messages = Agent(llm, TOOLS, max_steps=4).run(step)
        observation = (messages[-1]["content"] or "").strip()
        observations.append(observation)
        print(step)
        print(observation, "\n")
    return observations


observations = run_plan(plan)

Os três arquivos foram lidos, um por passo. Cada agente recebeu uma instrução curta, com uma ferramenta óbvia para ela, e nenhum deles precisou decidir o que vinha depois.

A terceira observação mostra o ruído típico deste modelo: ela troca o nome do arquivo por `marcado.txt` e acrescenta uma frase que não estava no arquivo lido. O número que ela informa está correto, e é só o número que interessa na etapa seguinte.

### Replanejamento

O plano foi montado a partir da pasta e executado logo depois, supondo que nada mudasse no meio do caminho. Apagar um relatório com o plano já pronto quebra essa suposição.

In [ ]:
old_plan = plan_from_files()      # plano montado com os três arquivos
(REPORTS / "marco.txt").unlink()  # a pasta muda depois de o plano ficar pronto

old_observations = run_plan(old_plan)

Os dois primeiros passos funcionaram e o terceiro voltou com um erro de arquivo inexistente, porque a sequência tinha sido montada antes da remoção.

Quem decide o que fazer com esse erro é o programa, com uma pergunta ao modelo. O esquema `Decision` fecha a resposta em duas opções, e é isso que permite usá-la em um `if`.

In [ ]:
class Decision(BaseModel):
    action: Literal["continuar", "replanejar"]


DECISION_PROMPT = """Tarefa: {task}
Última observação: {observation}

O plano ainda serve, ou é melhor replanejar?"""


def should_replan(observation: str) -> Decision:
    """Pergunta ao modelo se o plano ainda serve, depois de uma observação."""
    prompt = DECISION_PROMPT.format(task=TASK, observation=observation)
    return llm.generate_structured([{"role": "user", "content": prompt}], Decision, max_tokens=30)


decision = should_replan(old_observations[-1])
decision

In [ ]:
if decision.action == "replanejar":
    old_plan = plan_from_files()   # replanejar é montar o plano de novo, com o que existe agora

pd.DataFrame({"passo": old_plan.steps})

O plano novo tem dois passos, porque a pasta tem dois arquivos, e o passo impossível saiu antes de ser tentado outra vez.

Replanejar é refazer o plano a partir do mundo como ele está agora. Um agente que replaneja a cada observação, porém, nunca termina, então o número de replanejamentos costuma ser um limite explícito do programa.

In [ ]:
(REPORTS / "marco.txt").write_text(
    f"Relatorio mensal\nTotal de entregas: {MONTHS['marco.txt']}\n", encoding="utf-8"
)
print("pasta restaurada:", list_files())

### Soma em Python

Faltam as duas últimas etapas da tarefa, somar e gravar. As três observações trazem os números escritos dentro de frases, e somar em linguagem natural é onde este modelo erra.

O trabalho se divide mais uma vez. Ao modelo cabe achar o número dentro de cada frase, que é leitura de texto, e o esquema `Totals` garante que o que volta é uma lista de inteiros. A soma é feita por Python, que não erra conta.

In [ ]:
class Totals(BaseModel):
    values: list[int]


TOTALS_PROMPT = """Extraia o total de entregas de cada anotação. São {count} anotações, então devolva {count} números.

{notes}"""


def extract_totals(observations: list[str]) -> Totals:
    """Extrai um inteiro de cada observação, para que a soma saia de Python."""
    notes = "\n".join(f"Anotação {number}: {text}" for number, text in enumerate(observations, start=1))
    prompt = TOTALS_PROMPT.format(count=len(observations), notes=notes)
    return llm.generate_structured([{"role": "user", "content": prompt}], Totals, max_tokens=120)

In [ ]:
totals = extract_totals(observations)
answer = f"Total de entregas: {sum(totals.values)}"

print("números extraídos:", totals.values, "| soma:", sum(totals.values))
print(write_file("total.txt", answer + "\n"))
print(read_file("total.txt"))

Os três inteiros vieram validados pelo esquema, a soma foi feita em Python e `total.txt` recebeu o valor certo, igual ao gabarito criado no início.

Foi essa divisão de trabalho que resolveu a tarefa em que o ReAct parou. O modelo leu os arquivos e achou os números; o programa contou os passos e fez a conta.

### Exercício 2

A secretaria copia para a pasta o relatório de `abril.txt`, com o valor que você quiser, e junto um `avisos.txt`, que não é relatório e não tem total de entregas. Refaça o caminho da seção com a pasta nova, de `plan_from_files` até a gravação, e responda quantos passos o plano teve, o que o agente respondeu no passo do `avisos.txt` e o que isso fez com o total gravado; depois mude `plan_from_files` para planejar apenas os relatórios.

In [ ]:
# Seu código aqui

## Reflexão

As duas seções anteriores tratam de chegar à resposta. A reflexão trata do que fazer com ela depois de pronta: uma chamada avalia o que saiu, e outra reescreve.

Perguntar se a resposta está boa devolve elogio. Uma crítica que serve ao programa precisa de três coisas, e as três estão no esquema `Critique`: uma rubrica, que aqui é precisão e completude; uma lista de problemas concretos, que alimenta a reescrita; e um veredito fechado em `accept` ou `revise`, que decide se há outra rodada.

In [ ]:
class Critique(BaseModel):
    score: int = Field(ge=0, le=5)
    issues: list[str]
    verdict: Literal["accept", "revise"]


CRITIQUE_PROMPT = """Tarefa: {task}
Resposta: {answer}

Dê uma nota de 0 a 5 para precisão e completude, liste os problemas concretos e decida entre accept e revise."""


def critique(answer: str) -> Critique:
    """Avalia uma resposta por precisão e completude, com nota e veredito."""
    prompt = CRITIQUE_PROMPT.format(task=TASK, answer=answer)
    return llm.generate_structured([{"role": "user", "content": prompt}], Critique, max_tokens=250)

In [ ]:
weak_answer = "A soma das entregas dá uns três mil, mais ou menos."

review = critique(weak_answer)
print("nota:", review.score, "| veredito:", review.verdict)
for issue in review.issues:
    print(" -", issue)

A nota e a lista de problemas são valores, então o programa decide com eles, em vez de mostrar um texto ao usuário.

O conteúdo da crítica mostra o limite da técnica neste modelo: dois dos três apontamentos falam do enunciado da tarefa em vez da resposta, e nenhum deles diz o que mais importa, que o total exato está faltando.

### Laço de revisão

A crítica vira laço quando a resposta reescrita volta ao crítico. São duas saídas possíveis: o veredito `accept`, quando o crítico se dá por satisfeito, e o limite de rodadas, para o caso de ele nunca se dar.

In [ ]:
REWRITE_PROMPT = """Tarefa: {task}
Resposta anterior: {answer}
Problemas encontrados: {issues}

Escreva uma resposta melhor."""


def revise(answer: str, max_rounds: int = 2) -> tuple[str, list[dict]]:
    """Critica e reescreve a resposta até o veredito accept ou o fim das rodadas."""
    history = []

    for number in range(1, max_rounds + 1):
        review = critique(answer)
        history.append({"rodada": number, "nota": review.score, "veredito": review.verdict})
        if review.verdict == "accept":
            break

        prompt = REWRITE_PROMPT.format(task=TASK, answer=answer, issues=review.issues)
        answer = llm.invoke([{"role": "user", "content": prompt}], max_tokens=150)

    return answer, history

In [ ]:
final_answer, history = revise(weak_answer)

print(final_answer[:250])
pd.DataFrame(history)

A nota subiu de 3 para 4 e o veredito virou aceitação. A resposta, no entanto, piorou: a reescrita trocou a estimativa por um roteiro de como a tarefa seria feita, e o único número que sobrou é a numeração dos passos desse roteiro.

A explicação está em quem participa do laço. Nenhuma das chamadas de `revise` tem acesso aos arquivos, então a crítica apenas reorganiza o que já está no contexto, e o dado que falta continua faltando.

É a diferença entre as duas técnicas da aula. Planejar muda a estrutura do problema, tirando do modelo a contagem e a soma; refletir trabalha sobre o texto pronto, e serve para forma, coerência e verificação de critérios, não para repor uma leitura que ninguém fez.

### Exercício 3

O total vai ser enviado por e-mail à coordenação, então a resposta precisa trazer o número em algarismos e citar os arquivos que foram somados. Reescreva `CRITIQUE_PROMPT` com essas duas exigências, rode `revise` sobre `weak_answer` e critique também a resposta certa que a seção anterior gravou em `total.txt`, respondendo em qual rodada o veredito mudou e o que o crítico disse de cada uma das duas respostas.

In [ ]:
# Seu código aqui